# 01 — Load and Explore the Vivameda Longitudinal Sample

This notebook walks through the basic structure of the Vivameda 503-company longitudinal sample.

We'll load the dataset, inspect the schema, look at coverage across eras, and plot a single company's headcount evolution across 70 years.

The full dataset is available on Hugging Face: [Vivameda/longitudinal_503companies_1950_2020](https://huggingface.co/datasets/Vivameda/longitudinal_503companies_1950_2020)


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load directly from Hugging Face, or from local CSV if you've downloaded it
HF_URL = "https://huggingface.co/datasets/Vivameda/longitudinal_503companies_1950_2020/resolve/main/vivameda_longitudinal_sample_503companies_1950_2020.csv"

df = pd.read_csv(HF_URL)
print(f"Loaded {len(df):,} rows across {df['company_id'].nunique()} companies")
print(f"Year range: {df['year'].min()} to {df['year'].max()}")
print(f"Columns: {df.shape[1]}")

## Schema overview

The dataset is structured at the company-year grain. Each row describes a single company in a single year. The schema has 34 columns covering identity, workforce, growth, tenure, role mix, capability mix, and signal flags.

Coverage layers onto the base panel as source density increases through time. The `record_depth` column tells you what's available in each row.

In [ ]:
print("=== RECORD DEPTH BY ERA ===")
era_depth = df.groupby(['record_depth']).size().sort_values(ascending=False)
print(era_depth)
print()
print("=== SAMPLE TIER COMPOSITION ===")
print(df.groupby('sample_tier')['company_id'].nunique())

## NULL semantics

NULL values in this dataset are meaningful. They reflect insufficient signal density at that point in time, not random missingness.

For example, role data only exists for years 2010-2020 because the source signal density wasn't sufficient before that. Capability data starts in 1990. Filter accordingly.

In [ ]:
# How does column completeness vary across eras?
def coverage_by_era(df):
    layers = {
        'Pre-1990 (panel only)': df[df['year'] < 1990],
        '1990-2009 (+ capability)': df[(df['year'] >= 1990) & (df['year'] < 2010)],
        '2010-2016 (+ role)': df[(df['year'] >= 2010) & (df['year'] < 2017)],
        '2017-2020 (full record)': df[df['year'] >= 2017],
    }
    cols_to_check = ['headcount_observed', 'avg_tenure_years', 'top_capability_1', 'primary_role_bucket', 'early_scaling_flag']
    out = {}
    for era_name, era_df in layers.items():
        out[era_name] = {c: f"{era_df[c].notna().mean()*100:.0f}%" for c in cols_to_check}
    return pd.DataFrame(out).T

coverage_by_era(df)

## A single company across 70 years

Let's pick IBM and look at its observable history. The 70-year anchor companies in this sample are observable across the full 1950-2020 range, which lets us see how a single organization evolved through multiple economic regimes.

In [ ]:
ibm = df[df['company_name'] == 'IBM'].sort_values('year')
print(f"IBM: {len(ibm)} years observed, headcount range {ibm['headcount_observed'].min():,} to {ibm['headcount_observed'].max():,}")
ibm[['year', 'headcount_observed', 'growth_rate_yoy', 'growth_bucket', 'primary_role_bucket']].head(15)

In [ ]:
# Plot IBM's headcount evolution
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ibm['year'], ibm['headcount_observed'], linewidth=2, color='#2B6CB0')
ax.set_xlabel('Year')
ax.set_ylabel('Observed headcount')
ax.set_title('IBM observed headcount, 1950–2020')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Modern showcase tier

The dataset includes 11 modern high-profile companies (NVIDIA, Stripe, Databricks, Figma, etc.) for evaluators who want to see how the schema applies to recent scaling stories.

In [ ]:
showcase = df[df['sample_tier'] == 'modern_showcase']
print("Modern showcase companies:")
for name in showcase['company_name'].unique():
    rows = showcase[showcase['company_name'] == name]
    yr_range = f"{rows['year'].min()}-{rows['year'].max()}"
    peak = rows['headcount_observed'].max()
    print(f"  {name:25s} {yr_range}  peak headcount: {peak:,}")

## Industry distribution

The sample spans 75 normalized industry categories. Coverage is broad enough to support cross-industry comparisons.

In [ ]:
top_industries = df['industry'].value_counts().head(15)
print(top_industries)

## Next

In `02_compute_growth_signals.ipynb` we show how to derive custom signals from the raw growth fields, and how the four built-in signal flags (early_scaling, contraction, recovery, growth_acceleration) are computed.

For the full 4.2M company universe (1950-2020, 48M company-year records), see [vivameda.com](https://vivameda.com).
